# Random Forest Volatility Backtest

CPU-only tree-based walk-forward backtest. sklearn's RandomForestRegressor
handles missing values via surrogate splits, so no scaling or imputation needed.

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow tqdm

import sys
sys.path.insert(0, "/content/harxhar/colab")

In [ ]:
# ── Parameters ──
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
PERIODS_PER_DAY = 48
REFIT_FREQUENCY = 5
DATA_PATH = "all30min"

In [ ]:
# ── Load + transform ──
import numpy as np
import pandas as pd

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH, allow_missing=True)
print(f"Loaded: {df.shape}")

# Full transform on RV target (diurnal + winsor)
adj_rv, baseline = robust_transform(
    df, "RV", is_target=True, use_diurnal=True, winsor_window=240
)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"adj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# ── HAR features + DOW + hour ──
from src.transforms import resolve_har_lags, generate_har_features, add_calendar_features

df, har_names = generate_har_features(df, target_col="adj_RV")
cal_names = add_calendar_features(df)
feature_names = har_names + cal_names

print(f"HAR lags: {resolve_har_lags()}")
print(f"Features ({len(feature_names)}): {feature_names}")

# Drop initial NaN rows from HAR computation
max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)
print(f"Shape after lag trim: {df.shape}")

In [ ]:
# ── Random Forest model + walk-forward backtest ──
from sklearn.ensemble import RandomForestRegressor
from src.transforms import apply_horizon_shift
from src.scaling import run_backtest

# Extract arrays
X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

# Horizon shift
X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Samples: {len(y)}, Features: {X.shape[1]}, Train window: {train_win}")

model_fn = lambda: RandomForestRegressor(
    n_estimators=500, max_depth=10, min_samples_leaf=5,
    n_jobs=-1,
)

predictions = run_backtest(model_fn, X, y, train_win=train_win, refit_frequency=REFIT_FREQUENCY, use_scaling=False)
print(f"Predictions: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}")

In [ ]:
# ── Feature importance ──
import matplotlib.pyplot as plt

# Refit final model to get importances
from src.scaling import RollingBuffer

buf = RollingBuffer(train_win, X.shape[1])
for i in range(len(y) - train_win, len(y)):
    buf.add(X[i], y[i:i+1])
X_buf, y_buf = buf.get_view()
model = RandomForestRegressor(n_estimators=500, max_depth=10, min_samples_leaf=5, n_jobs=-1)
model.fit(X_buf, y_buf.ravel())

importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx])
ax.set_xlabel("Feature Importance (impurity decrease)")
ax.set_title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# ── Quick eval ──
from src.evaluation import apply_duan_smearing, calculate_metrics

oos_start = train_win
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(predictions, y_oos, baselines_oos)

results_df = pd.DataFrame({
    "date": dates_oos,
    "horizon": HORIZON,
    "true_adj": y_oos,
    "pred_adj": predictions,
    "true_raw": true_raw,
    "pred_raw": pred_raw,
})

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

In [ ]:
# export
"""Random Forest volatility backtest executor."""

import argparse
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from src.loading import load_raw_data
from src.transforms import (
    robust_transform,
    resolve_har_lags,
    generate_har_features,
    add_calendar_features,
    apply_horizon_shift,
    PERIODS_PER_DAY,
)
from src.scaling import run_backtest
from src.evaluation import apply_duan_smearing, calculate_metrics

In [ ]:
# Smoke-test imports land in this notebook's namespace.
assert callable(load_raw_data)
assert callable(run_backtest)
assert callable(apply_duan_smearing) and callable(calculate_metrics)
assert RandomForestRegressor().__class__.__name__ == "RandomForestRegressor"
assert isinstance(PERIODS_PER_DAY, int) and PERIODS_PER_DAY > 0
print("imports OK")

In [ ]:
# export
def main() -> None:
    parser = argparse.ArgumentParser(description="Random Forest walk-forward backtest")
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument("--train-window", type=int, default=500, help="training window in days")
    parser.add_argument("--start", type=int, default=0)
    parser.add_argument("--end", type=int, default=-1)
    parser.add_argument("--output-file", required=True)
    parser.add_argument("--params-file", default=None, help="JSON file with tuned hyperparams")
    args = parser.parse_args()

    tuned_params = {}
    if args.params_file:
        with open(args.params_file) as f:
            tuned_params = json.load(f)

    train_win_periods = args.train_window * PERIODS_PER_DAY

    df = load_raw_data(args.data_path, allow_missing=True)
    adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    df, har_names = generate_har_features(df, target_col="adj_RV")
    cal_names = add_calendar_features(df)
    feature_names = har_names + cal_names

    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, args.horizon)

    start = args.start
    end = len(X) if args.end == -1 else args.end
    X_chunk, y_chunk = X[start:end], y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines[start:end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(f"train_window ({train_win_periods} periods) >= chunk size ({len(X_chunk)})")

    def model_fn():
        defaults = dict(n_estimators=500, max_depth=10, min_samples_leaf=5, n_jobs=-1)
        defaults.update(tuned_params)
        return RandomForestRegressor(**defaults)

    preds = run_backtest(model_fn, X_chunk, y_chunk, train_win=train_win_periods, refit_frequency=5, use_scaling=False)

    oos_start = train_win_periods
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame({
        "date": dates_oos, "horizon": args.horizon,
        "true_adj": y_oos, "pred_adj": preds,
        "true_raw": true_raw, "pred_raw": pred_raw,
    })

    out_dir = os.path.dirname(args.output_file) or "."
    os.makedirs(out_dir, exist_ok=True)
    results.to_csv(args.output_file, index=False)

    metrics = calculate_metrics(results)
    metrics_path = os.path.join(out_dir, f"metrics_chunk_{start}.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f)
    print(f"Saved {len(results)} rows -> {args.output_file}")

In [ ]:
# Small safe test for main(): sanity-check its CLI surface without running a full backtest.
import inspect
sig = inspect.signature(main)
assert list(sig.parameters) == [], "main() takes no positional args"

# Parse a throwaway argv to confirm the argparse wiring matches expectations.
_p = argparse.ArgumentParser(description="Random Forest walk-forward backtest")
_p.add_argument("--data-path", default="all30min")
_p.add_argument("--horizon", type=int, default=1)
_p.add_argument("--train-window", type=int, default=500)
_p.add_argument("--start", type=int, default=0)
_p.add_argument("--end", type=int, default=-1)
_p.add_argument("--output-file", required=True)
_p.add_argument("--params-file", default=None)
_ns = _p.parse_args(["--output-file", "/tmp/rf_smoke.csv", "--horizon", "1"])
assert _ns.output_file == "/tmp/rf_smoke.csv" and _ns.horizon == 1 and _ns.end == -1
print("main() CLI wiring OK")

In [ ]:
# export
if __name__ == "__main__":
    main()

In [ ]:
from _exporter import export_notebook; export_notebook("ml_random_forest.ipynb", "../src/ml_random_forest.py")